# Module 5 - Lab: Generate, Check and Store Data with phyphox

This 45-minute lab follows the early data pipeline from the Module 5 impulse: **generate, check, organise, store**.

The goal is to turn a fresh phyphox export into data that another person can find, understand, and reuse later.

## Timebox
- 10 min: Generate a measurement and export it.
- 10 min: Check whether the recording worked correctly.
- 15 min: Organise files, rename them, and separate raw from preprocessed data.
- 10 min: Store the artefact in hot storage and document the location.

## Module 5 ideas used in this lab
- A raw export is not yet research data.
- Check directly after generation, before analysis begins.
- The check asks whether the run recorded correctly, not whether the result is nice.
- Keep or delete failed runs deliberately.
- Raw data is sacred: never edit it in place.
- Use clear folders, naming conventions, and versions.
- Store hot data in Sciebo and document the path.

## Section 1: Import Libraries

Run this cell first. The notebook uses Python to inspect file formats, check exported data, and create a small storage checklist.

In [ ]:
# Section 1: Import Libraries
from pathlib import Path
import json
import shutil
import sys

import numpy as np
import pandas as pd

project_root = Path.cwd()
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

from metadata_loader import (
    apply_recorded_data_path_override,
    load_metadata_context,
    save_public_metadata,
    summarize_metadata_context,
)
from data_format_loader import load_recorded_data, summarize_loaded_data

pd.set_option('display.max_columns', 30)
pd.set_option('display.precision', 4)

print('Libraries imported successfully.')

## Section 2: Generate the Measurement

Use phyphox on your phone to record one run on the model.

Before exporting, write down the context. This is the information that makes the file understandable later.

- What subsystem did you measure: `drivetrain` or `suspension`?
- Which quantity did you record: acceleration, light, rotation, etc.?
- Which phone/sensor did you use?
- What was the intended duration?
- Which run number is this?
- Did anything unusual happen during the run?

Export the phyphox data as CSV or Excel and place the raw export somewhere under `data/`.

## Section 3: Point to the Recorded Data

`metadata.json` stays compact. It records which file is used, which measurement type was performed, and a few naming/storage fields used later in the lab.

For the example data, switch between:

- `data/drivetrain/Example/Raw Data.csv`
- `data/drivetrain/Example/Raw Data.xlsx`
- `data/suspension/Example/Beschleunigung ohne g.xls`

In [ ]:
# Section 3: Point to the Recorded Data
metadata_context = load_metadata_context(project_root)
metadata = metadata_context['public_metadata']
metadata_path = metadata_context['public_metadata_path']

recorded_data_path_override = None
measurement_type_override = None
# recorded_data_path_override = 'data/drivetrain/Example/Raw Data.csv'
# measurement_type_override = 'drivetrain'
# recorded_data_path_override = 'data/suspension/Example/Beschleunigung ohne g.xls'
# measurement_type_override = 'suspension'

metadata = apply_recorded_data_path_override(
    metadata,
    recorded_data_path_override=recorded_data_path_override,
    measurement_type_override=measurement_type_override,
)
selected_data_path = project_root / metadata['recorded_data_path']

save_metadata = False
if save_metadata:
    save_public_metadata(metadata, project_root)
    print('Saved metadata:', metadata_path)

print('Recorded data path:', selected_data_path)
print('File exists:', selected_data_path.exists())
display(pd.json_normalize(summarize_metadata_context({
    **metadata_context,
    'public_metadata': metadata,
    'selected_data_path': selected_data_path,
}), sep='.').T.rename(columns={0: 'value'}))

## Section 4: Detect File Format and Load Data

One common data-quality problem is inconsistent format: comma vs. semicolon, decimal point vs. decimal comma. This section detects the format before loading.

Supported formats:
- Excel
- CSV comma, decimal point
- CSV tabulator, decimal point
- CSV semicolon, decimal point
- CSV tabulator, decimal comma
- CSV semicolon, decimal comma

In [ ]:
# Section 4: Detect File Format and Load Data
loaded_recorded_data = load_recorded_data(selected_data_path, project_root)
df_raw = loaded_recorded_data['table']
detected_format = loaded_recorded_data['format']

print('Detected format:', detected_format['format_label'])
print('Rows, columns:', df_raw.shape)
display(df_raw.head())
display(pd.DataFrame([summarize_loaded_data(loaded_recorded_data)]).T.rename(columns={0: 'value'}))

## Section 5: Extract Metadata Stored with the Data

This is the separation used in this course:

- `metadata.json`: only points to the recorded data file.
- The recorded data export or its neighbouring `meta/` folder contains the collection metadata.

For CSV, this cell checks the adjacent `meta/` folder. For Excel, it lists workbook sheets and previews their content.

In [ ]:
# Section 5: Extract Metadata Stored with the Data
recorded_metadata = loaded_recorded_data['recording_metadata']
display(pd.json_normalize(recorded_metadata, sep='.').T.rename(columns={0: 'value'}))

## Section 6: Check Correct Recording

This is a **recording check**, not an analysis result.

Ask the Module 5 questions:

- Did the file actually write?
- Is the duration plausible?
- Are the expected columns and axes present?
- Are there missing values, flat lines, clipping, or dropouts?
- Did the generation run as set up?

In [ ]:
# Section 6: Check Correct Recording
numeric_columns = df_raw.select_dtypes(include=[np.number]).columns.tolist()
time_candidates = [c for c in numeric_columns if 'time' in c.lower() or 'zeit' in c.lower()]
time_column = time_candidates[0] if time_candidates else None

quality_checks = []
quality_checks.append({'check': 'file_exists', 'result': selected_data_path.exists(), 'note': str(selected_data_path)})
quality_checks.append({'check': 'has_rows', 'result': len(df_raw) > 0, 'note': f'{len(df_raw)} rows'})
quality_checks.append({'check': 'has_columns', 'result': len(df_raw.columns) > 0, 'note': f'{len(df_raw.columns)} columns'})
quality_checks.append({'check': 'missing_values', 'result': not df_raw.isna().any().any(), 'note': str(df_raw.isna().sum().to_dict())})
quality_checks.append({'check': 'duplicate_rows', 'result': df_raw.duplicated().sum() == 0, 'note': f"{int(df_raw.duplicated().sum())} duplicate rows"})

for col in numeric_columns:
    unique_count = df_raw[col].nunique(dropna=True)
    quality_checks.append({
        'check': f'not_flat_line:{col}',
        'result': unique_count > 1,
        'note': f'{unique_count} unique values'
    })

if time_column:
    time_diff = df_raw[time_column].diff().dropna()
    duration = df_raw[time_column].max() - df_raw[time_column].min()
    quality_checks.append({'check': 'time_increases', 'result': bool((time_diff > 0).all()), 'note': f'time column: {time_column}'})
    quality_checks.append({'check': 'duration_available', 'result': bool(duration > 0), 'note': f'duration: {duration}'})

quality_report = pd.DataFrame(quality_checks)
display(quality_report)

display(df_raw.describe(include='all').T)

## Section 7: Keep or Delete Decision

Decide immediately after the check.

- Keep the run if it recorded correctly.
- Delete or repeat the run if it carries no useful information, for example wrong sensor, wrong axis, missing file, or malfunction.
- Exception: keep a failed run only if it documents a new challenge that should be discussed.

**Decision:** keep / repeat / delete

**Reason:** 

## Section 8: Organise Raw, Preprocessed, Analysed, Docs

Use a shallow structure that another person can navigate.

Recommended structure:

```text
data/<subsystem>/<run_id>/
  raw/
  preprocessed/
  analysed/
  docs/
```

Raw data is sacred. Never edit it in place.

In [ ]:
# Section 8: Organise Raw, Preprocessed, Analysed, Docs
subsystem = metadata.get('measurement_type', 'drivetrain')
quantity = metadata.get('quantity', 'measurement')
run_name = metadata.get('run_name', 'Example')
run_number = 1
date_iso = '2026-07-01'
project = 'rdm'
stage = metadata.get('data_stage', 'raw')
version = metadata.get('version', 'v0.1.0')

run_id = f'{date_iso}_{project}_{subsystem}_{quantity}_run{run_number:02d}'
target_root = project_root / 'data' / subsystem / run_id
raw_folder = target_root / 'raw'
preprocessed_folder = target_root / 'preprocessed'
analysed_folder = target_root / 'analysed'
docs_folder = target_root / 'docs'

for folder in [raw_folder, preprocessed_folder, analysed_folder, docs_folder]:
    folder.mkdir(parents=True, exist_ok=True)

recommended_filename = f'{run_id}_{stage}_{version}{selected_data_path.suffix.lower()}'
target_raw_path = raw_folder / recommended_filename

print('Run name:', run_name)
print('Run ID:', run_id)
print('Recommended raw filename:', recommended_filename)
print('Target raw path:', target_raw_path)

## Section 9: Copy Raw Export Without Editing It

This creates a copy of the raw export with a clear name. The original export is not modified.

Only set `copy_raw_file = True` after checking the target path.

In [ ]:
# Section 9: Copy Raw Export Without Editing It
copy_raw_file = False

if copy_raw_file:
    if target_raw_path.exists():
        raise FileExistsError(f'Target file already exists. Choose a new version: {target_raw_path}')
    shutil.copy2(selected_data_path, target_raw_path)
    print('Copied raw export to:', target_raw_path)
else:
    print('Dry run only. Set copy_raw_file = True to copy the raw export.')

## Section 10: Create a Documentation File

A README explains the rules in plain words. This one records what was checked, where the data came from, and what storage path should be documented in the DMP.

In [ ]:
# Section 10: Create a Documentation File
sciebo_path = metadata.get('hot_storage_path') or 'sciebo://CHANGE_ME/project/path/to/hot-storage'

lab5_summary = {
    'recorded_data_path': str(selected_data_path.relative_to(project_root)).replace('\\', '/'),
    'measurement_type': subsystem,
    'run_name': run_name,
    'quantity': quantity,
    'data_stage': stage,
    'version': version,
    'detected_format': detected_format,
    'metadata_source': recorded_metadata.get('source'),
    'recommended_run_id': run_id,
    'recommended_raw_path': str(target_raw_path.relative_to(project_root)).replace('\\', '/'),
    'row_count': int(df_raw.shape[0]),
    'column_count': int(df_raw.shape[1]),
    'columns': df_raw.columns.tolist(),
    'quality_checks': quality_report.to_dict(orient='records'),
    'hot_storage_path': sciebo_path,
    'backup_note': 'Hot data should be synced, backed up, shared with the team, and documented in the DMP.'
}

readme_text = f'''# {run_id}

## What was measured
- Measurement type: {subsystem}
- Quantity: {quantity}
- Run name: {run_name}
- Run number: {run_number}
- Source file: {lab5_summary['recorded_data_path']}
- Detected format: {detected_format['format_label']}

## Data handling
- Raw data is stored separately and must not be edited in place.
- Preprocessed and analysed files are new artefacts and need their own names and versions.
- File names use ISO date, project, subsystem, quantity, run number, stage, and version.

## Hot storage
- Sciebo path: {sciebo_path}
- Document this path in the DMP.

## Recording check
{quality_report.to_string(index=False)}
'''

summary_path = docs_folder / f'{run_id}_lab5_summary.json'
readme_path = docs_folder / 'README.md'

write_docs = False
if write_docs:
    with summary_path.open('w', encoding='utf-8') as f:
        json.dump(lab5_summary, f, indent=2)
    readme_path.write_text(readme_text, encoding='utf-8')
    print('Wrote summary:', summary_path)
    print('Wrote README:', readme_path)
else:
    print('Dry run only. Set write_docs = True to write documentation files.')
    print(readme_text[:1200])

## Section 11: Hot Storage Checklist

Use Sciebo as hot storage for active working data.

- Synced and backed up: a laptop failure must not cost a run.
- Shared with the team: no email attachments as the main data store.
- Path documented: the location goes into the DMP.
- 3-2-1 thinking: three copies, two storage types, one elsewhere. In this course, Sciebo covers the backup side for hot data.

**Sciebo path documented in DMP:** yes / no

**Who can access the folder:**

**Remaining action before analysis:**

## Section 12: Final Lab Check

Before leaving Lab 5, make sure you can answer these questions:

- Did the run record correctly?
- Did you keep, repeat, or delete it for a documented reason?
- Is raw data separate from preprocessed or analysed data?
- Does the filename follow the agreed naming convention?
- Is the version explicit, for example `v0.1.0`?
- Is the Sciebo hot-storage path documented?
- Can another group find this file tomorrow and understand what it is?

The stored artefact is now ready to become input for the following analysis modules.